# Data Preparation

In [1]:
# 分析数据结构
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [2]:
# 检查数据相关信息
import pandas as pd

train_raw = dataset["train"].to_pandas()
test_raw = dataset["test"].to_pandas()
unsupervised_raw = dataset["unsupervised"].to_pandas()

print("train:", train_raw.shape)
display(train_raw.head())
display(train_raw.info())

print("test:", test_raw.shape)
display(test_raw.head())
display(test_raw.info())

print("unsupervised:", unsupervised_raw.shape)
display(unsupervised_raw.head())
display(unsupervised_raw.info())

train: (25000, 2)


,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    25000 non-null  object
 1   label   25000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 390.8+ KB


None

test: (25000, 2)


...

None

unsupervised: (50000, 2)


,text,label
0,This is just a precious little diamond. The pl...,-1
1,When I say this is my favourite film of all ti...,-1
2,I saw this movie because I am a huge fan of th...,-1
3,Being that the only foreign films I usually li...,-1
4,After seeing Point of No Return (a great movie...,-1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    50000 non-null  object
 1   label   50000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 781.4+ KB


None

In [3]:
# 保存原始数据

from sentiment_analysis.paths import RAW_DIR

RAW_DIR.mkdir(parents=True, exist_ok=True)

train_raw.to_csv(RAW_DIR / "train.csv", index= False, encoding= "utf-8")
test_raw.to_csv(RAW_DIR / "test.csv", index= False, encoding= "utf-8")
unsupervised_raw.to_csv(RAW_DIR / "unsupervised.csv", index= False, encoding= "utf-8")

## 划分训练集、测试集、验证集

注意到在原始数据中，官方已经对数据集进行了划分。因此，我们这里选择与官方保持一致，仅从训练集中划分出 10% 作为验证集。

In [4]:
# 划分训练集，测试集，验证集

from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_raw,
    test_size= 0.1,
    train_size= 0.9,
    random_state= 42,               # 固定随机数种子，保证分层结果可复现
    stratify=train_raw["label"],    # 根据标签进行分层抽样
)

In [5]:
# 检查划分后的数据集数量，正负样本分布是否合理

splits = {
    "train": train_df,
    "validation": val_df,
    "test": test_raw
}

rows = []

for name, data in splits.items():
    value_cout = data["label"].value_counts()
    rows.append({
        "NAME": name,
        "COUNT": len(data),
        "NEGATIVE": value_cout.get(0, 0),
        "POSITIVE": value_cout.get(1, 0),
        "NEGATIVE RATIO": value_cout.get(0, 0) / len(data),
        "POSITIVE RATIO": value_cout.get(1, 0) / len(data),
    })

summary = pd.DataFrame(rows)
display(summary)

,NAME,COUNT,NEGATIVE,POSITIVE,NEGATIVE RATIO,POSITIVE RATIO
0,train,22500,11250,11250,0.5,0.5
1,validation,2500,1250,1250,0.5,0.5
2,test,25000,12500,12500,0.5,0.5


In [6]:
# 保存划分后的数据集
from sentiment_analysis.paths import INTERIM_DIR

INTERIM_DIR.mkdir(parents= True, exist_ok= True)

train_df.to_csv(INTERIM_DIR / "train.csv", index= False, encoding= "utf-8")
val_df.to_csv(INTERIM_DIR / "val.csv", index= False, encoding= "utf-8")
test_raw.to_csv(INTERIM_DIR / "test.csv", index= False, encoding= "utf-8")